# Prediction Preview -- Dataset509_ARCADE_1x1_4c (4-class)

Adapted from `analysis/501_ARCADE/preview_results.ipynb` (the binary-objective
preview notebook) for the current 4-class objective (LAD/RCA/LCX/LM on
`Dataset509_ARCADE_1x1_4c`). Reuses that folder's `segmentation_topology.py`
loading helpers (`image_files`/`case_id_from_image`/`load_class_id_mask`/
`resize_mask_to` are dataset-agnostic already -- `load_class_id_mask` grew a
`binarize=False` mode specifically so multi-class callers like this one and
`compression/collect_results.py` can get real per-pixel class IDs instead of
a collapsed foreground/background mask).

Shows raw image / ground truth / prediction / confusion-overlay panels for
one model at a time -- purely visual, not a metrics run (per-class Dice/
clDice for every trained config already lives in `compression/results.csv`
via `collect_results.py`; this notebook is for *looking* at predictions, not
recomputing numbers that already exist).

Defaults to **S8-ReLU** (`nnUNetTrainerENet_8_2_relu`, stage
`8_reginterleaved_isolation`) -- the best dice in the whole compression sweep
so far (0.8218, vs. U4's 0.7471 reference and S6-RegInterleaved's 0.8167,
the config it's a plain-ReLU ablation of). Change `NET_NAME` in the setup
cell (+ add an entry to `_MODEL_CONFIGS` if it's not already there) to
preview a different checkpoint.

## Setup

In [ ]:

# Imports and paths
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from PIL import Image


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "enet").exists() and (candidate / "data").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd())
PACKAGE_ROOT = REPO_ROOT / "enet" if (REPO_ROOT / "enet").exists() else REPO_ROOT

# segmentation_topology.py's loading helpers are dataset-agnostic (no
# Dataset501-specific paths) -- reused as-is rather than reimplemented here.
sys.path.insert(0, str(REPO_ROOT / "analysis" / "501_ARCADE"))
import segmentation_topology as topo

RESULTS_DIR = REPO_ROOT / "compression" / "notebook" / "preview_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "Dataset509_ARCADE_1x1_4c"
DATASET_ID = "509"
CONFIGURATION = "2d"
PLANS_NAME = "nnUNetPlans"
FOLD = 0
TRAINER_CLASS = "nnUNetTrainerENet"

DATASET_DIR = REPO_ROOT / "data" / "nnUNet_raw" / DATASET_NAME
IMAGES_TS_DIR = DATASET_DIR / "imagesTs"
LABELS_TS_DIR = DATASET_DIR / "labelsTs"
NNUNET_PREPROCESSED = REPO_ROOT / "data" / "nnUNet_preprocessed"
NNUNET_RESULTS = REPO_ROOT / "data" / "nnUNet_results"

# Every ENET_* env var build_network_architecture needs to reconstruct the
# checkpoint's exact architecture before loading weights -- same convention
# as compression/collect_results.py / the stage_*.job scripts. Add more
# entries here to preview other trained Dataset509 checkpoints.
_MODEL_CONFIGS = {
    "5_1_dscnoprojection_dense_dilation": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1",
    ),
    "1_naive_baseline_U4": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="default",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="1", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="0",
    ),
    "1_naive_baseline_Baseline": dict(
        ENET_CHANNELS="16,64,128,64,16", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="default",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="1", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="0",
    ),
    "4_8_dense_dilation": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1",
    ),
    "3_transfer_original": dict(
        ENET_CHANNELS="16,64,128,64,16", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="max_unpool", ENET_CONTEXT_PATTERN="default",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="1", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1",
    ),
    "6_1_reg_interleaved": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1",
    ),
    "6_2_schedule_a": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_a",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1",
    ),
    "7_1_reginterleaved_e1shape": dict(
        ENET_CHANNELS="4,16,28,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1",
    ),
    "7_2_reginterleaved_extrainitial": dict(
        ENET_CHANNELS="8,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1",
    ),
    "8_4_dscnoproj_context_only_dsc": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="1",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1", ENET_DSC_NO_PROJECTION_CONTEXT_ONLY="1",
    ),
    "8_1_dscnoproj_context_only": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1", ENET_DSC_NO_PROJECTION_CONTEXT_ONLY="1",
    ),
    "8_2_relu": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="0", ENET_DSC_NO_PROJECTION="1",
    ),
    "8_3_reg_bookend_dsc": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="1", ENET_REG_BOOKEND_DSC="1",
    ),
    "9_1_dense_dilation_dsc_projected": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="1",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="0",
    ),
    "9_2_dense_dilation_dsc_projected_extra_depth": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="1",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="0",
    ),
    "9_3_dense_dilation_reg_trailing": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,10,10,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_trailing",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="1",
        ENET_USE_PRELU="1", ENET_DSC_NO_PROJECTION="0",
    ),
    "9_4_dense_dilation_dsc_projected_relu": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,8,8,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="1",
        ENET_USE_PRELU="0", ENET_DSC_NO_PROJECTION="0",
    ),
    "10_1_reginterleaved_separable_projected": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="1", ENET_SEPARABLE_DILATED="1", ENET_DSC_NO_PROJECTION="0",
    ),
    "10_2_reginterleaved_separable_projected_relu": dict(
        ENET_CHANNELS="4,16,32,16,4", ENET_BOTTLENECKS="4,11,11,2,1",
        ENET_DECODER_TYPE="upsample_conv", ENET_CONTEXT_PATTERN="dense_dilation_reg_interleaved",
        ENET_USE_DILATED="1", ENET_USE_ASYMMETRIC="0", ENET_USE_STRIDED="1", ENET_USE_DSC="0",
        ENET_USE_PRELU="0", ENET_SEPARABLE_DILATED="1", ENET_DSC_NO_PROJECTION="0",
    ),
}
NET_NAME = "8_2_relu"   # <-- change this (+ rerun) to switch models -- current sweep leader (dice=0.8218)
MODEL_ENV = _MODEL_CONFIGS[NET_NAME]

CHECKPOINT_PATH = (
    NNUNET_RESULTS / DATASET_NAME / f"{TRAINER_CLASS}_{NET_NAME}__{PLANS_NAME}__{CONFIGURATION}"
    / f"fold_{FOLD}" / "checkpoint_best.pth"
)
CHECKPOINT_NAME = CHECKPOINT_PATH.name
PREDICTION_DIR = DATASET_DIR / f"labelsPr_{TRAINER_CLASS}_{NET_NAME}"

# Dataset509_ARCADE_1x1_4c: 0=background, 1=LAD, 2=RCA, 3=LCX, 4=LM
# (dataset.json's "labels" dict).
N_CLASSES = 5
CLASS_LABELS = ["Background", "LAD", "RCA", "LCX", "LM"]
CLASS_COLORS = ["#000000", "#e6194b", "#3cb44b", "#4363d8", "#f58231"]
CMAP = mcolors.ListedColormap(CLASS_COLORS)
NORM = mcolors.BoundaryNorm(range(N_CLASSES + 1), N_CLASSES)
IMSHOW_MASK_KW = dict(cmap=CMAP, norm=NORM, interpolation="nearest")
IMG_EXTS = topo.IMG_EXTS

print("Repo root       :", REPO_ROOT)
print("Dataset dir     :", DATASET_DIR)
print("Prediction dir  :", PREDICTION_DIR)
print("Checkpoint      :", CHECKPOINT_PATH)
print("Model env       :", MODEL_ENV)

## Optional: Run Inference From a Checkpoint

Set `RUN_INFERENCE = True` to (re)generate predictions. Defaults to
`False` here because `collect_results.py`'s own test-split inference run
already populated `labelsPr_{TRAINER_CLASS}_{NET_NAME}` for every trained
config referenced in `_MODEL_CONFIGS`, including the current default
(S8-ReLU) -- flip it on for a checkpoint that hasn't been inferenced yet.

In [ ]:
RUN_INFERENCE = False          # Set True to (re)generate predictions from CHECKPOINT_PATH
OVERWRITE_PREDICTIONS = False  # Set True to replace an existing PREDICTION_DIR
DEVICE = "cuda"
DISABLE_TTA = True
NUM_PREPROCESS_WORKERS = 2
NUM_EXPORT_WORKERS = 2


def run_nnunet_inference() -> None:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
    if not IMAGES_TS_DIR.exists():
        raise FileNotFoundError(f"imagesTs not found: {IMAGES_TS_DIR}")
    if PREDICTION_DIR.exists() and OVERWRITE_PREDICTIONS:
        shutil.rmtree(PREDICTION_DIR)
    PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env["nnUNet_raw"] = str(REPO_ROOT / "data" / "nnUNet_raw")
    env["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
    env["nnUNet_results"] = str(NNUNET_RESULTS)
    env.update(MODEL_ENV)

    model_folder = NNUNET_RESULTS / DATASET_NAME / f"{TRAINER_CLASS}_{NET_NAME}__{PLANS_NAME}__{CONFIGURATION}"
    command = [
        shutil.which("nnUNetv2_predict_from_modelfolder") or "nnUNetv2_predict_from_modelfolder",
        "-i", str(IMAGES_TS_DIR), "-o", str(PREDICTION_DIR), "-m", str(model_folder),
        "-f", str(FOLD), "-chk", CHECKPOINT_NAME, "-device", DEVICE,
        "-npp", str(NUM_PREPROCESS_WORKERS), "-nps", str(NUM_EXPORT_WORKERS),
        "--disable_progress_bar",
    ]
    if DISABLE_TTA:
        command.append("--disable_tta")

    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=PACKAGE_ROOT, env=env, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"nnU-Net inference failed with return code {completed.returncode}")


if RUN_INFERENCE:
    run_nnunet_inference()
elif PREDICTION_DIR.exists():
    print(f"RUN_INFERENCE=False; reusing existing predictions in {PREDICTION_DIR}")
else:
    print(f"RUN_INFERENCE=False but {PREDICTION_DIR} doesn't exist -- set RUN_INFERENCE=True.")

## Build Sample Index

In [ ]:
img_files = topo.image_files(IMAGES_TS_DIR)
gt_files = {p.stem: p for p in topo.image_files(LABELS_TS_DIR)}
pred_files = {p.stem: p for p in topo.image_files(PREDICTION_DIR)} if PREDICTION_DIR.exists() else {}

samples = []
for img in img_files:
    cid = topo.case_id_from_image(img)
    if cid in gt_files and cid in pred_files:
        samples.append({"case_id": cid, "image": img, "gt": gt_files[cid], "pred": pred_files[cid]})

print(f"ImagesTs found   : {len(img_files)}")
print(f"LabelsTs found   : {len(gt_files)}")
print(f"Predictions found: {len(pred_files)}")
print(f"Matched samples  : {len(samples)}")

if not samples:
    raise FileNotFoundError(
        "No matched samples found. Run inference first or check PREDICTION_DIR.\n"
        f"ImagesTs: {IMAGES_TS_DIR}\nLabelsTs: {LABELS_TS_DIR}\nPredictions: {PREDICTION_DIR}"
    )

## Headline numbers (from `compression/results.csv`)

Already-computed per-class Dice for this checkpoint, for context before
looking at individual predictions -- not recomputed here. `dice` is the
equally-weighted mean of the 4 per-class Dice scores; `dice_binary` collapses
every foreground class to one "vessel" label before scoring (all-classes-
merged, the same convention the retired binary Dataset501 objective used).
The two can diverge noticeably: `dice` gives the LM class -- rare enough to
be absent from most cases, so it collects many free "both masks empty" 1.0
scores -- equal 1/4 weight in the mean, which tends to pull `dice` above
`dice_binary` for the stronger configs (see e.g. S6-RegInterleaved:
dice=0.8167 vs. dice_binary=0.7519). `dice_binary` is blind to class-
confusion errors (predicting the wrong vessel branch still counts as
"vessel", so it can't separate a genuinely blurry class boundary from a
clean one), so neither number alone tells the whole story -- read them
together.</cell id="cell-7">


In [ ]:
results_csv = REPO_ROOT / "compression" / "results.csv"
if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    row = results_df[results_df["config_name"] == f"{TRAINER_CLASS}_{NET_NAME}"]
    if not row.empty:
        display(row[["config_name", "stage", "params", "flops", "dice", "dice_binary",
                      "dice_LAD", "dice_RCA", "dice_LCX", "dice_LM", "converged_flag"]])
    else:
        print(f"No results.csv row for {TRAINER_CLASS}_{NET_NAME} yet.")
else:
    print(f"{results_csv} not found.")

## Loading Helpers

In [ ]:
def load_sample(idx: int):
    item = samples[idx]
    raw = np.asarray(Image.open(item["image"]).convert("L"), dtype=np.float32) / 255.0
    gt = topo.load_class_id_mask(item["gt"], binarize=False)
    pred = topo.load_class_id_mask(item["pred"], binarize=False)
    pred = topo.resize_mask_to(pred, gt.shape)
    if raw.shape != gt.shape:
        raw = np.asarray(
            Image.fromarray((raw * 255).astype(np.uint8)).resize((gt.shape[1], gt.shape[0]), Image.BICUBIC),
            dtype=np.float32,
        ) / 255.0
    return raw, gt, pred


def legend_handles():
    return [mpatches.Patch(color=CLASS_COLORS[i], label=f"{i}: {CLASS_LABELS[i]}") for i in range(N_CLASSES)]

## Preview: Raw Image, Ground Truth, Prediction

In [ ]:
def show_sample(idx: int, ax_row=None, save_path: Path | None = None):
    raw, gt, pred = load_sample(idx)
    standalone = ax_row is None
    if standalone:
        fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    else:
        axes = ax_row
        fig = axes[0].figure

    axes[0].imshow(raw, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
    axes[1].imshow(gt, **IMSHOW_MASK_KW); axes[1].set_title("Ground truth"); axes[1].axis("off")
    axes[2].imshow(pred, **IMSHOW_MASK_KW); axes[2].set_title("Prediction"); axes[2].axis("off")

    if standalone:
        fig.legend(handles=legend_handles(), loc="lower center", ncol=N_CLASSES, bbox_to_anchor=(0.5, -0.04), frameon=False)
        case_id = samples[idx]["case_id"]
        fig.suptitle(f"{idx}: {case_id}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        if save_path:
            fig.savefig(save_path, bbox_inches="tight", dpi=150)
        plt.show()


def show_batch(indices=None, n_samples: int = 6, save_path: Path | None = None):
    if indices is None:
        indices = list(range(min(n_samples, len(samples))))
    indices = list(indices)
    fig, axes = plt.subplots(len(indices), 3, figsize=(14, 4.2 * len(indices)))
    axes = np.atleast_2d(axes)
    for row, idx in enumerate(indices):
        show_sample(idx, ax_row=axes[row])
        axes[row][0].set_ylabel(samples[idx]["case_id"], fontsize=9, rotation=90, labelpad=8)
    fig.legend(handles=legend_handles(), loc="lower center", ncol=N_CLASSES, bbox_to_anchor=(0.5, -0.01), frameon=False)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()

show_batch(n_samples=6, save_path=RESULTS_DIR / f"{NET_NAME}_batch_preview.png")

## Confusion Overlay

Generalized from the binary notebook's TP/FP/FN scheme to 4 foreground
classes: a pixel is only "correct" (green) if both foreground AND the same
vessel class match GT -- two different vessel classes overlapping counts as
its own **class confusion** category (magenta), not a plain TP, since
"right vessel, wrong branch" is a materially different failure mode from
"missed vessel" or "hallucinated vessel" on this task.

In [ ]:
CONFUSION_COLORS = {
    "Correct":   np.array([0, 200, 0, 210], dtype=np.uint8),    # gt fg == pred fg, same class
    "ClassConf": np.array([200, 0, 200, 210], dtype=np.uint8),  # both fg, different class
    "FP":        np.array([220, 0, 0, 200], dtype=np.uint8),    # pred fg, gt background
    "FN":        np.array([255, 190, 0, 210], dtype=np.uint8),  # gt fg, pred background
}


def make_confusion_overlay(gt: np.ndarray, pred: np.ndarray) -> np.ndarray:
    gt_fg = gt > 0
    pred_fg = pred > 0
    same_class = gt == pred
    overlay = np.zeros((*gt.shape, 4), dtype=np.uint8)
    overlay[gt_fg & pred_fg & same_class] = CONFUSION_COLORS["Correct"]
    overlay[gt_fg & pred_fg & ~same_class] = CONFUSION_COLORS["ClassConf"]
    overlay[~gt_fg & pred_fg] = CONFUSION_COLORS["FP"]
    overlay[gt_fg & ~pred_fg] = CONFUSION_COLORS["FN"]
    return overlay


CONFUSION_LEGEND = [
    mpatches.Patch(color=CONFUSION_COLORS["Correct"] / 255, label="Correct: right vessel class"),
    mpatches.Patch(color=CONFUSION_COLORS["ClassConf"] / 255, label="Class confusion: vessel, wrong branch"),
    mpatches.Patch(color=CONFUSION_COLORS["FP"] / 255, label="FP: predicted vessel on background"),
    mpatches.Patch(color=CONFUSION_COLORS["FN"] / 255, label="FN: missed GT vessel"),
]


def show_confusion(idx: int, save_path: Path | None = None):
    raw, gt, pred = load_sample(idx)
    overlay = make_confusion_overlay(gt, pred)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(raw, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
    axes[1].imshow(gt, **IMSHOW_MASK_KW); axes[1].set_title("Ground truth"); axes[1].axis("off")
    axes[2].imshow(pred, **IMSHOW_MASK_KW); axes[2].set_title("Prediction"); axes[2].axis("off")
    axes[3].imshow(raw, cmap="gray"); axes[3].imshow(overlay); axes[3].set_title("Confusion"); axes[3].axis("off")
    fig.legend(handles=CONFUSION_LEGEND, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.08), frameon=False)
    case_id = samples[idx]["case_id"]
    fig.suptitle(f"{idx}: {case_id}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()

show_confusion(0, save_path=RESULTS_DIR / f"{NET_NAME}_confusion_sample0.png")

In [ ]:
def _confusion_grid(indices, save_path: Path | None = None):
    indices = list(indices)
    if not indices:
        print("No samples to display.")
        return
    fig, axes = plt.subplots(len(indices), 4, figsize=(20, 4.5 * len(indices)))
    axes = np.atleast_2d(axes)
    for col, title in enumerate(["Raw", "Ground truth", "Prediction", "Confusion"]):
        axes[0, col].set_title(title, fontsize=11, fontweight="bold")
    for row, idx in enumerate(indices):
        raw, gt, pred = load_sample(idx)
        overlay = make_confusion_overlay(gt, pred)
        case_id = samples[idx]["case_id"]
        axes[row, 0].imshow(raw, cmap="gray")
        axes[row, 0].set_ylabel(f"#{idx}\n{case_id}", fontsize=8, rotation=0, labelpad=34, va="center")
        axes[row, 0].axis("off")
        axes[row, 1].imshow(gt, **IMSHOW_MASK_KW); axes[row, 1].axis("off")
        axes[row, 2].imshow(pred, **IMSHOW_MASK_KW); axes[row, 2].axis("off")
        axes[row, 3].imshow(raw, cmap="gray"); axes[row, 3].imshow(overlay); axes[row, 3].axis("off")
    fig.legend(handles=CONFUSION_LEGEND, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.01), frameon=False)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved: {save_path}")
    plt.show()


def show_confusion_batch(n_samples: int = 6, save_path: Path | None = None):
    n_samples = min(n_samples, len(samples))
    _confusion_grid(range(n_samples), save_path=save_path)


def show_confusion_indices(indices, save_path: Path | None = None):
    _confusion_grid(indices, save_path=save_path)

show_confusion_batch(n_samples=8, save_path=RESULTS_DIR / f"{NET_NAME}_confusion_grid.png")

## Random N-Sample Comparison Across Configs

Unlike every cell above (which previews whatever `NET_NAME` is currently
active), this compares several configs at once on the SAME randomly-chosen
cases -- `random.seed(SEED)` before sampling makes the case list
reproducible and identical across every config in `COMPARE_NET_NAMES`, so
row `i` in one config's grid is the same case as row `i` in another's.
Reuses each config's own `labelsPr_` predictions directly (no re-inference).

In [ ]:
import random

COMPARE_NET_NAMES = ["6_1_reg_interleaved", "6_2_schedule_a", "1_naive_baseline_U4"]
N_RANDOM = 15
SEED = 42


def load_sample_for(net_name: str, item: dict):
    """Same loading logic as load_sample(), but for an arbitrary (not
    necessarily the currently-active NET_NAME's) config's prediction dir --
    the single-NET_NAME `samples`/`load_sample` globals above only cover
    whichever model is currently loaded."""
    raw = np.asarray(Image.open(item["image"]).convert("L"), dtype=np.float32) / 255.0
    gt = topo.load_class_id_mask(item["gt"], binarize=False)
    pred = topo.load_class_id_mask(item["pred"], binarize=False)
    pred = topo.resize_mask_to(pred, gt.shape)
    if raw.shape != gt.shape:
        raw = np.asarray(
            Image.fromarray((raw * 255).astype(np.uint8)).resize((gt.shape[1], gt.shape[0]), Image.BICUBIC),
            dtype=np.float32,
        ) / 255.0
    return raw, gt, pred


def show_confusion_random_multi(net_names: list[str] = COMPARE_NET_NAMES, n: int = N_RANDOM, seed: int = SEED):
    all_img_files = topo.image_files(IMAGES_TS_DIR)
    gt_files_all = {p.stem: p for p in topo.image_files(LABELS_TS_DIR)}
    case_ids = sorted({
        topo.case_id_from_image(img) for img in all_img_files
        if topo.case_id_from_image(img) in gt_files_all
    })
    random.seed(seed)
    chosen_case_ids = sorted(random.sample(case_ids, n))
    print(f"Chosen {n} case IDs (seed={seed}): {chosen_case_ids}")
    img_by_cid = {topo.case_id_from_image(img): img for img in all_img_files}

    for net_name in net_names:
        prediction_dir = DATASET_DIR / f"labelsPr_{TRAINER_CLASS}_{net_name}"
        pred_files = {p.stem: p for p in topo.image_files(prediction_dir)} if prediction_dir.exists() else {}
        items = [
            {"case_id": cid, "image": img_by_cid[cid], "gt": gt_files_all[cid], "pred": pred_files[cid]}
            for cid in chosen_case_ids if cid in pred_files and cid in img_by_cid
        ]
        if len(items) < n:
            print(f"{net_name}: only {len(items)}/{n} matched (missing predictions?) -- skipping.")
            continue

        row_g = results_df[results_df["config_name"] == f"{TRAINER_CLASS}_{net_name}"] if "results_df" in dir() else None
        dice_str = f"dice={row_g.iloc[0]['dice']:.4f}" if row_g is not None and not row_g.empty else "dice=?"

        fig, axes = plt.subplots(len(items), 4, figsize=(20, 4.5 * len(items)))
        axes = np.atleast_2d(axes)
        for col, title in enumerate(["Raw", "Ground truth", "Prediction", "Confusion"]):
            axes[0, col].set_title(title, fontsize=11, fontweight="bold")
        for row_i, item in enumerate(items):
            raw, gt, pred = load_sample_for(net_name, item)
            overlay = make_confusion_overlay(gt, pred)
            axes[row_i, 0].imshow(raw, cmap="gray")
            axes[row_i, 0].set_ylabel(item["case_id"], fontsize=8, rotation=0, labelpad=34, va="center")
            axes[row_i, 0].axis("off")
            axes[row_i, 1].imshow(gt, **IMSHOW_MASK_KW); axes[row_i, 1].axis("off")
            axes[row_i, 2].imshow(pred, **IMSHOW_MASK_KW); axes[row_i, 2].axis("off")
            axes[row_i, 3].imshow(raw, cmap="gray"); axes[row_i, 3].imshow(overlay); axes[row_i, 3].axis("off")
        fig.legend(handles=CONFUSION_LEGEND, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.005), frameon=False)
        fig.suptitle(f"{net_name} -- {dice_str} (random {n}, seed={seed})", fontsize=14, fontweight="bold", y=1.005)
        plt.tight_layout(rect=[0, 0.03, 1, 1])
        out_path = RESULTS_DIR / f"{net_name}_confusion_grid_random{n}.png"
        fig.savefig(out_path, bbox_inches="tight", dpi=150)
        plt.show()
        print(f"Saved: {out_path}")


show_confusion_random_multi()